# Gradients through a graph

> Run a graph forward once, then ask how its outputs move with every input,
> for every sample, in one backward sweep.

**Time:** ~10 min · **Runs on:** CPU (the same cells capture one CUDA graph
for the reverse when the planes are cupy arrays) · **You need:** eagle, hawk and NumPy; read first: {doc}`09_graph_model`

You will end with gradients checked against finite differences, a fork whose
two branches add up correctly, and a small optimisation loop that reuses one
reverse pass.

**Colab or Kaggle:** run this first cell (it installs the packages and downloads the helper files this notebook uses). Elsewhere it only checks the imports.

In [1]:
# Colab setup, generated by python -m raptor_docs.nb colab_setup (do not edit).
import os

try:
    import google.colab  # noqa: F401
    HOSTED = True  # a Colab runtime
except ImportError:
    HOSTED = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # a Kaggle kernel

if HOSTED:
    import importlib.metadata, pathlib, re, shutil, subprocess, sys, urllib.request

    gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
    extra = ""
    if gpu:
        # the CUDA major the runtime already has: a CuPy that is installed, else torch's, else the driver's
        major = None
        for m in ("13", "12"):
            try:
                importlib.metadata.version(f"cupy-cuda{m}x")
                major = m
                break
            except importlib.metadata.PackageNotFoundError:
                pass
        if major is None:
            try:
                import torch
                major = "13" if str(torch.version.cuda).startswith("13") else None
            except ImportError:
                pass
        if major is None:
            smi = re.search(r"CUDA Version: (\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
            major = "13" if smi and int(smi.group(1)) >= 13 else "12"
        extra = f"[cuda{major}]"
        print(f"GPU found: installing the {extra} extras to match this runtime.")
    pre = ["--pre"] if os.environ.get("RAPTOR_PRE") else []  # optional: test a release candidate
    ref = os.environ.get("RAPTOR_REF", "main")  # optional: fetch the helper files from this git ref
    if pre or "RAPTOR_REF" in os.environ:
        print(f"Pre-release test: installing with --pre, helper files from {ref}.")
    # our packages with --upgrade, so a runtime that kept an older release gets the current one
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pre, f"raptor-eagle{extra}", f"raptor-hawk{extra}"])
    if not gpu:
        print("Running on the CPU (no GPU in this runtime). For a GPU: " + ("Settings → Accelerator → GPU T4 x2." if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "Runtime → Change runtime type → T4 GPU."))

try:
    import eagle, hawk  # noqa: F401
except ImportError as err:
    raise ImportError(
        "eagle, hawk not found. Install with `pip install raptor-eagle raptor-hawk`. For a GPU, add [cuda12] (or [cuda13]) to raptor-eagle and raptor-hawk."
    ) from err

In [2]:
import tempfile

import numpy as np
import eagle

import hawk
from hawk import Mutable, Param, Scalar
from hawk import math as m

## Gradients from a builder

If you wrote the graph as a `@eagle.Graph.builder` function
({doc}`../graph_model` starts there), build it with `record=True` and
reverse it. Seeds and `wrt` are always explicit. A seed is `"result"` (or
`"result.key"` into a dict) or an array a kernel wrote through a `Mutable`;
`wrt` names builder arguments, or a kernel result you kept with `buffer=` (not one made inside a loop or branch).

In [3]:
import hawk
from hawk import Param, Scalar


@hawk.kernel
def affine(x: Scalar, a: Param, b: Param) -> Scalar:
    return a * x + b


@hawk.kernel
def sq(v: Scalar) -> Scalar:
    return v * v


@eagle.Graph.builder
def fit(x, record=False):
    g = eagle.Graph(record=record)
    a = affine(x, 2.0, 1.0, buffer=True)    # keep `a`: readable after a run, and a gradient key
    g.result = sq(a)
    return g


gr = fit(np.arange(4.0), record=True)
gr()                                          # a recorded graph runs with g() or launch()
grads = eagle.diff.reverse(gr.record, {"result": np.ones(4)}, wrt=("x", "a"))
print(grads["x"], grads["a"])

[ 4. 12. 20. 28.] [ 2.  6. 10. 14.]


`dJ/dx = 4 (2x + 1)` and `dJ/da = 2a`. Ask for a plain temporary (a kernel
result you did not keep) and the message tells you to give it a `buffer=`.
For a graph you built with `add`, the keys are `"<node name>.<plane>"`, as below.

## A graph to differentiate

Two small [hawk](https://amasat01.github.io/hawk/) kernels: `y = a x²`, then
`z = sin(y)`. `a` is a uniform: one number every sample shares. The only new
thing compared with {doc}`09_graph_model` is `build(record=True)`: it tells
the graph to keep what a reverse pass will need.

In [4]:
@hawk.kernel
def square(x: Scalar, a: Param, y: Mutable[Scalar]):
    y = a * x * x


@hawk.kernel
def wave(x: Scalar, y: Mutable[Scalar]):
    y = m.sin(x)


square_plan, wave_plan = eagle.deploy([square, wave], targets=("host",),
                                      cache_dir=tempfile.mkdtemp())
n = 6
x = np.linspace(-1.0, 1.0, n)
y, z = np.zeros(n), np.zeros(n)

g = (eagle.Graph()
     .add(square_plan.bind(x=x, a=0.5, y=y), name="square")
     .add(wave_plan.bind(x=y, y=z), name="wave")
     .build(record=True))
g.launch()
print(z)
print(g.record)

[0.47942554 0.17902957 0.01999867 0.01999867 0.17902957 0.47942554]
<eagle.diff.Record generation 1, live>


`g.record` is the {term}`Record` of that launch: what its reverse reads.
Its {term}`generation` is 1, because it is the first launch.

## The reverse pass

Say our objective is the plain sum of the outputs, `J = Σ zᵢ`. Reverse mode
({term}`reverse mode`) needs one thing from us: how `J` moves with each
output, `∂J/∂zᵢ`. That is the {term}`seed`, one value per sample, and for a
plain sum every seed is 1. (JAX calls it a cotangent, PyTorch
`grad_outputs`, CasADi `adj_seed`.)

Both nodes call their output `y`, so the output of `wave` is named
`"wave.y"`.

In [5]:
grads = eagle.diff.reverse(g.record, {"wave.y": np.ones(n)})
print(list(grads))
print("dJ/dx =", grads["x"])
print("dJ/da =", grads["a"])
print(grads.report)

['x', 'a']
dJ/dx = [-0.87758256 -0.59030622 -0.19996     0.19996     0.59030622  0.87758256]
dJ/da = 2.5435165831215194
Report(generation=1, bitwise={'x': True, 'a': False}, chunks=1)


Two {term}`gradient`s came back. `grads["x"]` has one value per sample, in
sample order, because `x` is a per-sample input. `grads["a"]` is one number:
`a` is shared, so each sample's contribution is summed. Nobody wrote a
derivative: eagle built each kernel's {term}`vjp` with `hawk.diff.vjp` and ran
them backwards, `wave` first, then `square`.

## Is it right?

`z = sin(a x²)`, so `∂z/∂x = cos(a x²) · 2 a x` and `∂J/∂a = Σ cos(a x²) x²`.
Check both, and also against central differences, which need nothing but
the forward:

In [6]:
a = 0.5
print(np.allclose(grads["x"], np.cos(a * x * x) * 2 * a * x, rtol=1e-12))
print(np.isclose(grads["a"], np.sum(np.cos(a * x * x) * x * x), rtol=1e-12))


def forward(x_, a_):
    y_, z_ = np.zeros(n), np.zeros(n)
    (eagle.Graph().add(square_plan.bind(x=x_, a=a_, y=y_))
     .add(wave_plan.bind(x=y_, y=z_)).build().launch())
    return z_


h = 1e-6
fd_x = (forward(x + h, a) - forward(x - h, a)) / (2 * h)
fd_a = np.sum(forward(x, a + h) - forward(x, a - h)) / (2 * h)
print(np.allclose(grads["x"], fd_x, rtol=1e-6), np.isclose(grads["a"], fd_a, rtol=1e-6))

True
True
True True


## Two branches reading one input

In a fork, two nodes read the same plane. Its {term}`adjoint` (how `J` moves
with it) is then the sum of what flows back from both branches. eagle adds
them in one fixed order, the order you added the branches, so the answer is
the same bits on every run.

In [7]:
@hawk.kernel
def join(u: Scalar, v: Scalar, w: Mutable[Scalar]):
    w = u * v + u


(join_plan,) = eagle.deploy([join], targets=("host",), cache_dir=tempfile.mkdtemp())
u, v, w = np.zeros(n), np.zeros(n), np.zeros(n)
fork = (eagle.Graph()
        .add_group([square_plan.bind(x=x, a=a, y=u), wave_plan.bind(x=x, y=v)],
                   names=["square", "wave"])
        .add(join_plan.bind(u=u, v=v, w=w), name="join")
        .build(record=True))
fork.launch()
fork.show()

gf = eagle.diff.reverse(fork.record, {"w": np.ones(n)})
want = (np.sin(x) + 1) * 2 * a * x + a * x * x * np.cos(x)   # both branches
print(np.allclose(gf["x"], want, rtol=1e-12))

eagle.Graph  3 nodes · 1 group · backend: cpu · recorded

  stage 0 branch_tasks ┌─────────┐   ┌───────┐
                       │ square  │   │ wave  │
                       └────┬────┘   └───┬───┘
                            y            y
                            └─────┬──────┘
  stage 1 work_share          ┌───┴───┐
                              │ join  │
                              └───────┘



True


(autodiff-record)=
## One Record per launch

A Record points at the graph's own planes, which the next launch overwrites.
So the graph will not record a new launch while the last Record is still
waiting for its reverse:

In [8]:
g2 = (eagle.Graph()
      .add(square_plan.bind(x=x.copy(), a=a, y=np.zeros(n)), name="square")
      .build(record=True))
g2.launch()
first = g2.record
try:
    g2.launch()
except RuntimeError as err:
    print(err)

kept = first.keep()        # a private copy: the graph is free to run again
g2.launch()
print(first.generation, kept.generation, g2.record.generation)
print(eagle.diff.reverse(kept, {"y": np.ones(n)})["x"])

eagle.Graph.launch: the Record of the previous launch is still live; reverse it, or record.keep() to keep a private copy
1 1 2
[-1.  -0.6 -0.2  0.2  0.6  1. ]


Reversing `first` now would be refused: its launch is no longer the newest,
and eagle will not hand back a gradient computed from a mix of two launches.
`kept` stays valid for good.

## Built once, run in a loop

`eagle.diff.reverse` builds a reverse per call. In a loop, build an
{class}`eagle.diff.Reverse` once and call it; on the GPU it is captured once
and replayed. Here we move each `x` so that `z = sin(a x²)` reaches a target,
by plain gradient descent on `J = ½ Σ (zᵢ - targetᵢ)²`, whose seed is
`z - target`. `wrt=["x"]` ({term}`wrt`) asks for just the gradient we use.

In [9]:
target = np.full(n, 0.3)
rev = eagle.diff.Reverse(g)
for step in range(200):
    g.launch()
    grads = rev(g.record, {"wave.y": z - target}, wrt=["x"])
    x -= 0.5 * np.asarray(grads["x"])      # the graph reads x where it lies: no rebuild
print(np.round(z, 6))

[0.3 0.3 0.3 0.3 0.3 0.3]


## An optimisation loop over a parameter

Now tune the shared parameter `a` instead, so that the average of `z` hits
0.2: `J = ½ (mean(z) - 0.2)²`, whose seed is `(mean(z) - 0.2) / n` for every
sample. `g.update(a=...)` changes `a` between launches. A recorded graph keeps
each parameter in a one-cell plane that the kernels and their vjps read, so
this is a copy into that cell: nothing is rebuilt, and on the GPU nothing is
recaptured, forward or reverse.

In [10]:
a_fit = 0.5
for step in range(100):
    g.update(a=a_fit)
    g.launch()
    mean = z.mean()
    grads = rev(g.record, {"wave.y": np.full(n, (mean - 0.2) / n)}, wrt=["a"])
    a_fit -= 2.0 * float(grads["a"])
print(round(a_fit, 6), round(z.mean(), 6))

0.330428 0.2


## A node you do not differentiate

Some nodes set the pace of the run and are not part of the objective: a controller that shrinks a step size, say. Add it with
`grad=False`. The reverse skips it, so it needs no vjp, and the gradient is the one the other kernels would give
without it. Below, `halve` halves a one-cell parameter `a` after `drag` has read it, every launch.

In [11]:
from hawk.diff import param_planes


@hawk.kernel
def drag(x: Scalar, a: Param[Scalar], y: Mutable[Scalar]):
    y = a * x * x


@hawk.kernel
def halve(a: Mutable[Scalar]):
    a = a * 0.5


drag_plan, halve_plan = eagle.plan.auto([param_planes(drag), halve], targets=("host",),
                                        cache_dir=tempfile.mkdtemp(), scalar_type="float64")
cell = np.full(1, 0.8)
controlled = (eagle.Graph()
           .add(drag_plan.bind(x=x, a=cell, y=np.zeros(n)), name="drag")
           .add(halve_plan.bind(a=cell), name="halve", grad=False)
           .build(record=True))
for launch in range(3):
    used = float(cell[0])  # the value drag reads on this launch
    controlled.launch()
    grads = eagle.diff.reverse(controlled.record, {"drag.y": np.ones(n)})
    print(f"launch {launch}: a read {used:.2f}, left {cell[0]:.2f}, gradient right:",
          bool(np.allclose(grads["x"], 2 * used * x)))

launch 0: a read 0.80, left 0.40, gradient right: True
launch 1: a read 0.40, left 0.20, gradient right: True
launch 2: a read 0.20, left 0.10, gradient right: True


(autodiff-bits)=
## Same bits, every time

Every sum eagle does on the way back has one fixed order, and a parameter's
gradient is summed in sample order, so a gradient is {term}`bitwise`
reproducible: the same bits for every thread count and every run.
`grads.report.bitwise` says so per plane; the only gradients that can say
`False` come from a kernel that reads a table by index, and `strict=True`
refuses those instead.

In [12]:
fork.launch()                     # x has moved since: record a fresh launch
r1 = eagle.diff.reverse(fork.record, {"w": np.ones(n)})
r2 = eagle.diff.reverse(fork.record, {"w": np.ones(n)})
print(np.asarray(r1["x"]).tobytes() == np.asarray(r2["x"]).tobytes(), r1.report.bitwise)

True {'x': True, 'a': False}


## What just happened

- `build(record=True)` made the graph keep what its reverse needs: your own
  planes, plus a copy of any plane a later node overwrites.
- `eagle.diff.reverse(g.record, seeds)` ran each node's vjp backwards and
  returned the gradient with respect to every input: per sample for per-sample
  inputs, summed for shared parameters.
- In a fork, the adjoints of the shared input were summed in a fixed order.
- A Record names one launch; `keep()` keeps a copy when you need it longer.
- `eagle.diff.Reverse` is the built-once form for loops.
- `grad=False` leaves a controller out of the reverse; no vjp is needed for it.

## Try this

- Seed `"wave.y"` with `2 * z`: that is the gradient of `J = Σ zᵢ²`.
- Pass `seeds={"x": ...}` and read the message: it tells you that `∂J/∂x`
  is read from the result, not seeded.
- Move the planes to the GPU (`cupy` arrays and `targets=("cuda",)`): the
  same cells capture the reverse into one CUDA graph.

## Limits

- <!-- limit: buffer-in-body-wrt --> A `buffer=` array made inside a device loop or branch is not a gradient key, because the body rewrites it every turn and no one array is its value (our scope): make the `buffer=` outside the body, or seed `result`.
- <!-- limit: record-loop-control --> A `record=True` loop body holds kernel launches only: a `break`, `continue`, `if`/`elif`/`else` or `match` inside it is refused (our scope): keep the control flow outside the recorded loop, or build with `record=False`.
- <!-- limit: record-uncapped-while --> An uncapped `while cell:` cannot be recorded with `record=True` (our scope): cap it with `for _ in range(N): if not cell: break`.
- <!-- limit: record-loop-two-names --> A recorded loop that reads one array under two names is refused (our scope): pass it under one name.

## Words used here

Several communities differentiate programs, and each has its own words for the
same few things. These are the ones eagle uses; the synonyms are there so you
can recognise what you already know.

A short glossary of these words (reverse mode, seed, adjoint, vjp and the rest)
lives in the {doc}`../../glossary`.


## Where the derivative kernels come from

Each launch node is reversed by its kernel's {term}`vjp`. For a plan made by
{func}`eagle.plan.auto` (or {func}`eagle.deploy`) from a hawk kernel, eagle
builds the vjp itself the first time it is needed, beside the kernel, and
keeps it. You write nothing.

For a plan built some other way (from a plugin you built with hawk by hand,
say), pass the vjp plan yourself:

```python

rev = eagle.diff.Reverse(g, vjps={"square": square_vjp_plan})

```

eagle checks that each one really is the vjp of that node's kernel and refuses
a mismatch, naming both kernels. A vjp is a kernel *of* the reverse, never a
step of a forward graph: ``eagle.Graph.add`` and ``eagle.plan.auto`` refuse
one.



(autodiff-budget)=
## Memory budget

Reversing a recorded {func}`eagle.until_done` or {func}`eagle.simulation`
(``record=True``) keeps the state at the start of every ``window`` steps and,
working one window at a time from the last, recomputes the window's steps and
steps the adjoint back through them. One sample therefore holds about
``ceil(max_steps / window) + window`` states: the default window,
``clamp(ceil(sqrt(max_steps)), 16, 1024)``, balances the two terms, and
``window="full"`` trades memory for no recompute.

``memory_budget`` (bytes; ``None``, the default, is half the free device
memory, or half the available host RAM) caps what the reverse holds. When the
whole batch does not fit, the reverse runs it in **passes** of ``256 * 2^q``
samples, the largest that fits, and combines the passes' parameter sums with
the canonical tree. Those sums are exactly the subtrees of the one-pass tree,
so **the gradients are the same bits at any budget** (see
{ref}`autodiff-bits`). What a smaller budget costs is time: each pass launches
its own reverse over fewer samples.

```python

sim = eagle.simulation(pendulum, ..., max_steps=400, record=True,
                       window=20, memory_budget=1_000_000)
sim.run()
grads = eagle.diff.reverse(sim.record, {"x": np.ones(n)})
grads.report.chunks                  # how many passes it took

```

A budget that cannot hold one sample, or fewer than the 256 a pass needs to keep
parameter gradients bitwise, is refused with the bytes and the fix: raise
``memory_budget``, lower ``max_steps``, or set ``window``. ``window=`` and
``memory_budget=`` without ``record=True`` are refused too.

``steps_reverse="per_step"`` (the default), ``"fused"`` or ``"auto"`` picks how a
``hawk.steps(kernel, K)`` run is reversed. ``"fused"`` sets the window to ``K`` and
reverses each window in one launch (the window's steps are re-run inside it); state
gradients match the per-step reverse bit for bit, parameter gradients to a few ULP.
``"auto"`` takes the choice from a measured table: today it picks ``"fused"`` on the
host for small batches (up to 64 samples), and ``"per_step"`` everywhere else, so
the GPU stays per-step until its rows are measured.



(autodiff-events)=
## Event refinement

A run-until-done stops a sample at the first *step* whose rule fires. Its stop
time is then a multiple of the step, and nudging an input moves it by nothing,
so the gradient of the stop time is zero. That is the right answer for the
stepped model, and the wrong one if the model is a continuous trajectory that
you sampled. ``events="refine"`` finds the crossing *inside* the stop step.

The kernel declares its time plane and step, ``@hawk.kernel(time="t",
time_step="dt")``, and its stop rule is read as a smooth guard ``g`` that
changes sign (``terminated = y < 0``; a count or a flag has no crossing and is
refused). The run then:

1. steps as usual, remembering the state before each sample's last step;
2. solves ``g = 0`` for the partial step ``tau`` in ``[0, dt]`` by Newton's
   method with a bisection fallback (at most 8 rounds);
3. writes the state *at* the crossing over the stepped one, and the event time
   and how it was found into ``runner.t_stop`` and ``runner.event_status``.

The reverse ({func}`eagle.diff.reverse` of a ``record=True`` run, no extra
flag) adds the **event sensitivity**: how the stop time moves with the
initial state and the parameters, from the implicit function theorem on
``g = 0``, together with its effect on the end state. Nothing else changes.

```python

sim = eagle.simulation(fall, dt=0.1, g=9.81, y=y0, v=v0, t=t0,
                       max_steps=200, events="refine", record=True)
sim.run()
sim.runner.t_stop                                    # the event time
grads = eagle.diff.reverse(sim.record, {"t": np.ones(n)})
grads["y"]                                           # dt*/dy0, here 1/(g t*)

```

A sample that never crosses inside its stop step (it hit ``max_steps``) is
refused, unless you pass ``on_no_event="zero"``: its stop stays the stepped
one, with zero event sensitivity. A grazing crossing, and two stop rules
crossing in one step, are always refused, because the sensitivity is not
defined there.

**Phases.** A hybrid trajectory (a ball that bounces, a pendulum that hits a
wall) is a *phase* graph: each continuous piece is a refined run, and
{func}`eagle.diff.phase` makes it a node of an ordinary {class}`eagle.Graph`,
with a reset kernel between pieces.

```python

graph = (eagle.Graph()
         .add_node(eagle.diff.phase(down), name="fall")
         .add(bounce_plan.bind(e=e, v_in=v_down, v=v_up), name="bounce")
         .add_node(eagle.diff.phase(up), name="rise")
         .build(record=True))
graph.launch()                       # restarts each phase, runs it to its event
grads = eagle.diff.reverse(graph.record, {"y": np.ones(n)})

```

Each launch of the graph restarts every phase and runs it to its event. The
reverse chains the phases through the reset kernels' vjps, which is the chain
rule across each jump (the saltation of the jump, in the hybrid-systems
literature). The number of phases is the graph's, so a phase inside
{func}`eagle.repeat_while` is refused. On the GPU a phase is an eager node (it
reads its own progress back between launches), and the graph is captured in
segments around it. The checked version of both examples is in
{doc}`11_conditionals_budgets_events`. A discontinuity in the model itself, a branch on a learnable
threshold, is trained through in {doc}`train_through_a_branch`.



(autodiff-conditionals)=
## Gradients through conditionals

A step inside {func}`eagle.skippable` runs only while its guard cell is
non-zero; one inside {func}`eagle.repeat_while` repeats while its guard holds,
up to a cap. Both reverse. The rule is the same for both: **the reverse
differentiates the launch that happened, and never evaluates a guard.**

- **Recorded.** ``build(record=True)`` saves the guard cell a ``skippable``
  read (a one-value ``uint32`` copy), and a ``repeat_while`` already writes
  its iteration count into a plane. These are all the reverse needs.
- **Replayed.** The reverse turns those saved values into flags it owns and
  runs the member's derivative only when the member ran: once for a taken
  ``skippable``, once per iteration, newest first, for a ``repeat_while``.
  The state each iteration started with is rebuilt from checkpoints kept every
  few iterations, on private copies, so your planes are never written.
  Flipping the guard between launches therefore needs no rebuild, and a
  guard that counts its own reads is read zero times by the reverse.
- **Passed through.** A skipped member leaves its planes as they were, so the
  adjoint of a plane it would have updated in place goes to the plane's
  earlier writer, and every other adjoint of the member is zero.

The sum over a ``repeat_while``'s iterations runs oldest first, the order of
the same chain written out as separate nodes, so the two give the same bits.

The checked version of the example is in {doc}`11_conditionals_budgets_events`; a branch on a learnable threshold is trained through in {doc}`train_through_a_branch`.


(autodiff-run-vjps)=
## Your model inside a recorded loop

A step list ``[node, plan]`` runs your model before the kernel on every step,
and a recorded run of it reverses with your vjp:

```python

node = eagle.graph.opaque(model_step, reads={"x": x, "v": v}, writes={"corr": corr},
                          name="model")
runner = eagle.until_done([node, plan], max_steps=1000, record=True, x=x, v=v, corr=corr)
runner.run()
grads = eagle.diff.reverse(runner.record, {"x": dJ_dx}, vjps={"model": vjp})

```

The node reads the plan's state and per-sample planes and writes planes the
plan reads (``eagle.simulate([node, kernel], ...)`` takes the same list). The
reverse re-runs it once per step while it recomputes a window, then calls
``vjp(primals, cotangents)`` once per step, host-synchronized, on the samples
still running at that step; your model's parameter gradients accumulate as
you write them. **Cost**: a hooked reverse costs what your model costs per
step; eagle's share is the un-hooked reverse (0.12 ms/step at 4k rows on a
P2000). **Memory**: one more window of each plane the node writes per sample.

The node must be pure, because the reverse re-runs it: the same inputs give
the same outputs (``model.eval()``, no random draw, no counter), row ``s``
of a write depends only on row ``s`` of the reads, it writes only its declared
planes, and the weights stay as they were until the reverse returns
(``run()``, ``reverse()``, then ``optimizer.step()``). eagle cannot see an
impure node; the test that catches it is yours: reverse twice, compare. The
vjp gets views of eagle's buffers, valid during the call: copy what you keep.

**HAZARD 3**: during the reverse the node's planes are eagle's scratch, and
they come back bitwise at the end. Do not read them while a reverse of that run
is in flight; a second reverse of it raises {class}`eagle.Busy`.

A node runs before the plan in a step: write the step as ``[node, plan]``. The run calls the
node on the initial state before step 0, and applying it once more after the last step is
yours; your framework's autograd of that call gives the seeds.


## Threads

A {class}`~eagle.diff.Reverse` owns its adjoint planes. Distinct ``Reverse``
objects run at the same time, also on the same Record (a reverse never writes
the Record), and give the same bits. One ``Reverse`` shared by two threads
raises {class}`eagle.Busy` in the second; give each thread its own.


## For C++ users

The plan of a reverse is plain C++: ``eagle::diff::transpose`` takes a graph
``Description`` ({doc}`../graph_view`) and, per node, the planes its vjp reads and
writes, and returns the order of the derivative steps, the summing joins and
the copies; ``eagle::diff::orderedSum`` is the fixed-order sum. See
{doc}`../../api/api_diff`.
